<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">

<a href="https://colab.research.google.com/github/yhilpisch/algocolab/blob/main/notebooks/02_deep_learning_gpu_trading.ipynb"
target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab"/></a>


# Algorithmic Trading with Python & Google Colab

## Session 2 — Teaching a Neural Network to Trade

### GPU Deep Learning and Honest Evaluation

The Python Quants GmbH | https://tpq.io<br>
© Dr. Yves J. Hilpisch | https://hilpisch.com

The live experiment follows one narrow, auditable path: reuse the Session 1
sample contract, fit a compact PyTorch classifier, select a trading threshold
on validation data, open the untouched test set once, and persist the complete
inference contract for Session 3.


## 1. Reconnect to the Named Drive Run

Paste the exact run ID printed by Session 1. The notebook never selects the
newest directory implicitly. In Colab, Drive is mounted at
`/content/drive/MyDrive/algo`; locally the same files are available below the
synced macOS path.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'src').is_dir():
            if (candidate / 'data' / 'eod_data.csv').is_file():
                return candidate
    raise FileNotFoundError(
        'Could not locate the companion repository root.'
    )
if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/algocolab')
    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                'git', 'clone', '--depth', '1',
                'https://github.com/yhilpisch/algocolab.git',
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    RUNS_ROOT = Path('/content/drive/MyDrive/algo/runs')
else:
    PROJECT_ROOT = find_project_root()
    RUNS_ROOT = Path(
        '/Users/yves/Google Drive/My Drive/algo/runs'
    )
RUN_ID = os.environ.get('WEBINAR_RUN_ID', '').strip()
if not RUN_ID:
    raise ValueError(
        'Set WEBINAR_RUN_ID to the exact Session 1 run identifier.'
    )
PERSIST_RESULTS = os.environ.get(
    'WEBINAR_PERSIST_RESULTS',
    str(IN_COLAB),
).lower() in {'1', 'true', 'yes'}
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Run: {RUN_ID}')

In [ ]:
from src.artifacts import RunBundle
from src.config import ExperimentConfig
from src.session2 import persist_session_two, run_session_two
bundle = RunBundle.open(
    RUNS_ROOT,
    RUN_ID,
    required_session=1,
)
config = ExperimentConfig(**bundle.manifest['configuration'])
session_one_metrics = bundle.path / 'session_1/strategy_metrics.csv'
print(f'Session 1 verified: {session_one_metrics}')
print(config)

## 2. What the Network Is—and Is Not—Testing

The classifier estimates the probability of a positive next-day EUR/USD
return from lagged returns, rolling volatility, and rolling momentum. It is a
non-linear alternative inside the prediction branch of algorithmic trading;
it says nothing about the viability of market making, execution, arbitrage,
or other non-directional strategies.


## 3. Build Features and Preserve Time Order

The feature matrix is built from information available before the target
return. Splitting remains chronological: the model never trains on future
observations.


In [ ]:
import pandas as pd
from src.data import create_lagged_features, load_eod_data
prices = load_eod_data(
    PROJECT_ROOT / 'data' / 'eod_data.csv',
    symbol=config.symbol,
)
prices = prices.loc[config.data_start:config.data_end]
features, target_return, target_direction = create_lagged_features(
    prices,
    lags=config.target_lags,
)
train_end = int(len(features) * config.train_ratio)
validation_end = int(
    len(features) * (config.train_ratio + config.validation_ratio)
)
x_train = features.iloc[:train_end]
x_validation = features.iloc[train_end:validation_end]
x_test = features.iloc[validation_end:]
y_train = target_direction.iloc[:train_end]
y_validation = target_direction.iloc[train_end:validation_end]
y_test = target_direction.iloc[validation_end:]
print(features.columns.tolist())
print(x_train.shape, x_validation.shape, x_test.shape)
x_train.head()

Scaling parameters belong to the training sample only. The same
training mean and scale are then applied to validation and test features.


In [ ]:
train_mean = x_train.mean()
train_scale = x_train.std(ddof=0).replace(0.0, 1.0)
x_train_scaled = (x_train - train_mean) / train_scale
x_validation_scaled = (x_validation - train_mean) / train_scale
x_test_scaled = (x_test - train_mean) / train_scale
print('Training mean:')
print(train_mean.round(6))
print('Training scale:')
print(train_scale.round(6))

In [ ]:
from torch.utils.data import DataLoader
from src.models import ModelConfig, TradingDataset, build_model
model_config = ModelConfig(
    input_dim=x_train_scaled.shape[1],
    hidden_units=(64, 32),
    dropout_rate=0.2,
)
model = build_model(model_config)
train_loader = DataLoader(
    TradingDataset(x_train_scaled, y_train),
    batch_size=64,
    shuffle=True,
)
parameter_count = sum(
    parameter.numel() for parameter in model.parameters()
)
print(model)
print(f'Trainable parameters: {parameter_count:,}')

The model maps the feature vector through two hidden layers.
Each hidden layer applies an affine transformation, batch normalization, a
rectified linear unit, and dropout:

\[
h_1=\operatorname{ReLU}(W_1x+b_1),\qquad
h_2=\operatorname{ReLU}(W_2h_1+b_2),\qquad
\hat p=\sigma(W_3h_2+b_3).
\]

Binary cross-entropy trains probability estimates. It does not directly
optimize return, drawdown, turnover, or Sharpe ratio.


In [ ]:
import time
import torch
from src.models import get_device
device = get_device()
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')
print(f'Accelerator: {device.type}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
def synchronize_device() -> None:
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()
matrix = torch.randn(1024, 1024, device=device)
synchronize_device()
started = time.perf_counter()
for _ in range(10):
    matrix @ matrix
synchronize_device()
elapsed = (time.perf_counter() - started) / 10
print(f'1,024 x 1,024 matmul: {elapsed:.4f}s')

## 4. Train One Compact Demonstration Model

All scaling parameters are fitted on the training sample only. The
chronological validation and test partitions remain later in time. The
canonical runner below repeats this visible pipeline and records the complete
evaluation contract.


In [ ]:
started = time.perf_counter()
results = run_session_two(
    PROJECT_ROOT / 'data' / 'eod_data.csv',
    config,
    epochs=20,
    hidden_units=(64, 32),
    dropout_rate=0.2,
    device=device,
)
elapsed = time.perf_counter() - started
parameter_count = sum(
    parameter.numel() for parameter in results.model.parameters()
)
print(f'Training time: {elapsed:.2f}s')
print(f'Trainable parameters: {parameter_count:,}')
print(results.model)
results.history.tail()

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
ax = results.history[['train_loss', 'val_loss']].plot(
    figsize=(9, 4),
    color=['#002D5A', '#2F80ED'],
)
ax.set(title='Training and validation loss', xlabel='Epoch')
ax.grid(alpha=0.2)
plt.show()

## 5. Select the Trading Rule on Validation Data

For a symmetric threshold \(\theta\), the position is long when the predicted
probability is greater than \(\theta\), short when it is less than
\(1-\theta\), and flat inside the deadband. We compare \(\theta\in\{0.50,
0.52,0.55\}\): for example, \(\theta=0.55\) means long above 55%, short below
45%, and no position between 45% and 55%. Thresholds are ranked only by
validation Sharpe after the canonical 0.5 basis point one-way cost. The test
sample is not used for this choice.


In [ ]:
threshold_columns = [
    'threshold',
    'active_fraction',
    'net_annual_return',
    'net_sharpe',
    'maximum_drawdown',
    'turnover_units',
]
results.threshold_results[threshold_columns]

In [ ]:
print(
    'Validation-selected threshold: '
    f'{results.threshold:.2f}'
)

## 6. Open the Test Set Once

The same test dates and cost convention are applied to the DNN, OLS,
momentum, random, and buy-and-hold baselines. Negative findings remain visible:
a flexible model is not evidence of stable alpha.


In [ ]:
test_metrics = results.strategy_metrics.query(
    "sample == 'test'"
)
test_metrics.set_index('strategy')[
    [
        'net_annual_return',
        'net_volatility',
        'net_sharpe',
        'maximum_drawdown',
        'turnover_units',
    ]
]

> **Research deepening beyond the live skeleton**
>
> - repeat across multiple neural-network and random-baseline seeds;
> - report uncertainty intervals and the full outcome distribution;
> - use walk-forward retraining and regime diagnostics;
> - correct for repeated model and feature searches;
> - enrich costs with slippage, financing, market impact, and capacity;
> - test alternative targets, horizons, architectures, and calibration.
>
> These extensions strengthen inference; they must not be used to search the
> untouched test sample for a better story.


## 7. Persist the Exact Inference Contract

The checkpoint contains the trained weights, exact architecture—including
dropout placement—feature order, training scaler, selected threshold, and run
ID. Session 3 refuses to proceed without this completed, checksummed bundle.


In [ ]:
if PERSIST_RESULTS:
    code_commit = subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    persist_session_two(
        bundle,
        results,
        code_commit=code_commit,
    )
    print(f'Session 2 persisted: {bundle.path}')
else:
    print('Persistence disabled; set WEBINAR_PERSIST_RESULTS=true to enable.')

## Session 2 Takeaway

Model capacity can discover non-linear patterns, but the measured test result
decides whether those patterns generalize economically. The durable output is
not a loose weight file: it is a validated inference contract ready for the
paper-trading simulation in Session 3.


---

<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">
